# Notebook 04: Seasonal Patterns

**One Sensor, One Year — Edition 1: India Breathes**

India's grid has a rhythm driven by three forces: **heat** (demand), **rain** (hydro/wind), and **sun** (solar). Let's map the seasons.

- Monthly averages by fuel type
- Seasonal regime shifts
- The monsoon handoff (coal → hydro + wind)
- Heatwave stress on the grid
- Winter coal dominance

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df24 = pd.read_csv('../data/processed/india_2024_derived.csv', parse_dates=['date'])
df24['month'] = df24['date'].dt.month
df24['month_name'] = df24['date'].dt.strftime('%b')
df24['day_of_year'] = df24['date'].dt.dayofyear

fuel_cols = ['coal', 'lignite', 'gas', 'nuclear', 'hydro', 'wind', 'solar', 'other_re']
fuel_labels = ['Coal', 'Lignite', 'Gas', 'Nuclear', 'Hydro', 'Wind', 'Solar', 'Other RE']
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

# Earth & Sky palette
palette = {
    'coal': '#3D2B1F', 'lignite': '#6B4226', 'gas': '#4A90D9',
    'nuclear': '#2EC4B6', 'hydro': '#1B4F72', 'wind': '#85C1E9',
    'solar': '#F5B041', 'other_re': '#A569BD',
}

## 1. Monthly Averages — Grouped Bar Chart

In [2]:
monthly = df24.groupby('month')[fuel_cols].mean()

fig = go.Figure()
for col, label in zip(fuel_cols, fuel_labels):
    fig.add_trace(go.Bar(
        x=month_names, y=monthly[col],
        name=label, marker_color=palette[col],
        hovertemplate=f'{label}: %{{y:.0f}} MU<extra></extra>',
    ))

fig.update_layout(
    title='Monthly Average Daily Generation by Fuel Type — India 2024',
    yaxis_title='Generation (MU/day)',
    barmode='group',
    plot_bgcolor='#FAFAF5', paper_bgcolor='#FAFAF5',
    height=500,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, x=0.5, xanchor='center'),
)
fig.show()

## 2. Monthly Stacked Bar — The Shifting Mix

In [3]:
fig = go.Figure()
for col, label in zip(fuel_cols, fuel_labels):
    fig.add_trace(go.Bar(
        x=month_names, y=monthly[col],
        name=label, marker_color=palette[col],
        hovertemplate=f'{label}: %{{y:.0f}} MU<extra></extra>',
    ))

fig.update_layout(
    title='Monthly Generation Mix (Stacked) — India 2024',
    yaxis_title='Generation (MU/day)',
    barmode='stack',
    plot_bgcolor='#FAFAF5', paper_bgcolor='#FAFAF5',
    height=500,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, x=0.5, xanchor='center'),
)
fig.show()

## 3. Monthly Normalized — The Share Shifts

In [4]:
monthly_pct = monthly.div(monthly.sum(axis=1), axis=0) * 100

fig = go.Figure()
for col, label in zip(fuel_cols, fuel_labels):
    fig.add_trace(go.Bar(
        x=month_names, y=monthly_pct[col],
        name=label, marker_color=palette[col],
        hovertemplate=f'{label}: %{{y:.1f}}%<extra></extra>',
    ))

fig.update_layout(
    title='Monthly Generation Share (%) — India 2024',
    yaxis_title='Share of Generation (%)',
    barmode='stack',
    plot_bgcolor='#FAFAF5', paper_bgcolor='#FAFAF5',
    height=500,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, x=0.5, xanchor='center'),
)
fig.show()

# Print the coal share by month
print('Coal share by month:')
for i, m in enumerate(month_names):
    print(f'  {m}: {monthly_pct["coal"].iloc[i]:.1f}%')

Coal share by month:
  Jan: 79.0%
  Feb: 77.7%
  Mar: 77.0%
  Apr: 76.6%
  May: 72.5%
  Jun: 70.7%
  Jul: 67.3%
  Aug: 65.6%
  Sep: 65.6%
  Oct: 73.4%
  Nov: 76.1%
  Dec: 75.9%


## 4. The Monsoon Handoff

When the monsoon arrives, coal retreats and hydro + wind step in. Let's quantify this handoff.

In [5]:
# Define seasons
seasons = {
    'Winter (Jan-Feb)': [1, 2],
    'Pre-Monsoon (Mar-May)': [3, 4, 5],
    'Monsoon (Jun-Sep)': [6, 7, 8, 9],
    'Post-Monsoon (Oct-Dec)': [10, 11, 12],
}

print('Seasonal Averages (MU/day):')
print('=' * 90)
print(f'{"Season":25s} {"Coal":>8s} {"Hydro":>8s} {"Wind":>8s} {"Solar":>8s} {"Total":>8s} {"Coal%":>7s} {"Clean%":>7s}')
print('-' * 90)

for name, months in seasons.items():
    mask = df24['month'].isin(months)
    s = df24[mask]
    coal_avg = s['coal'].mean()
    hydro_avg = s['hydro'].mean()
    wind_avg = s['wind'].mean()
    solar_avg = s['solar'].mean()
    total_avg = s['total'].mean()
    coal_pct = coal_avg / total_avg * 100
    clean = (s['nuclear'].mean() + hydro_avg + wind_avg + solar_avg + s['other_re'].mean())
    clean_pct = clean / total_avg * 100
    print(f'{name:25s} {coal_avg:8.0f} {hydro_avg:8.0f} {wind_avg:8.0f} {solar_avg:8.0f} {total_avg:8.0f} {coal_pct:6.1f}% {clean_pct:6.1f}%')

Seasonal Averages (MU/day):
Season                        Coal    Hydro     Wind    Solar    Total   Coal%  Clean%
------------------------------------------------------------------------------------------
Winter (Jan-Feb)              3583      208      145      300     4572   78.4%   17.6%
Pre-Monsoon (Mar-May)         3789      295      184      379     5040   75.2%   20.2%
Monsoon (Jun-Sep)             3402      597      343      332     5086   66.9%   28.7%
Post-Monsoon (Oct-Dec)        3401      336      122      342     4545   74.8%   21.7%


In [6]:
# Heatmap: fuel type vs month — normalized by each fuel's own max
monthly_norm = monthly.div(monthly.max(axis=0), axis=1)  # normalize each column 0-1

fig = go.Figure(go.Heatmap(
    z=monthly_norm[fuel_cols].values.T,
    x=month_names,
    y=fuel_labels,
    colorscale='YlOrRd',
    hovertemplate='%{y} in %{x}: %{z:.0%} of peak<extra></extra>',
    text=[[f'{v:.0%}' for v in row] for row in monthly_norm[fuel_cols].values.T],
    texttemplate='%{text}',
))

fig.update_layout(
    title='Seasonal Intensity Heatmap — Each Fuel Normalized to Its Own Peak Month',
    plot_bgcolor='#FAFAF5', paper_bgcolor='#FAFAF5',
    height=400,
)
fig.show()

print('\nReading the heatmap:')
print('  100% = the peak month for that fuel type')
print('  Hydro peaks in Aug-Sep (monsoon), drops to 25% of peak in Jan-Feb')
print('  Wind peaks in Jul-Aug (monsoon), drops to <15% of peak in winter')
print('  Coal peaks in May (heatwave demand), lowest in Aug (monsoon relief)')
print('  Solar is the steadiest — even its worst month is >70% of peak')


Reading the heatmap:
  100% = the peak month for that fuel type
  Hydro peaks in Aug-Sep (monsoon), drops to 25% of peak in Jan-Feb
  Wind peaks in Jul-Aug (monsoon), drops to <15% of peak in winter
  Coal peaks in May (heatwave demand), lowest in Aug (monsoon relief)
  Solar is the steadiest — even its worst month is >70% of peak


## 5. The Four Seasons of India's Grid

Radar/polar chart showing how the mix changes across seasons.

In [7]:
# Polar area chart — seasonal clean energy share
fig = go.Figure()

# Calculate monthly clean share and emissions intensity
monthly_total = df24.groupby('month')['total'].mean()
monthly_clean = df24.groupby('month')[['nuclear','hydro','wind','solar','other_re']].mean().sum(axis=1)
monthly_clean_pct = (monthly_clean / monthly_total * 100)
monthly_ei = df24.groupby('month')['emissions_intensity'].mean()

fig = make_subplots(rows=1, cols=2,
    specs=[[{'type': 'polar'}, {'type': 'polar'}]],
    subplot_titles=['Clean Energy Share (%)', 'Emissions Intensity (tCO2/GWh)'])

fig.add_trace(go.Scatterpolar(
    r=monthly_clean_pct.values,
    theta=month_names,
    fill='toself',
    name='Clean %',
    line_color='#27AE60',
    fillcolor='rgba(39,174,96,0.3)',
), row=1, col=1)

fig.add_trace(go.Scatterpolar(
    r=monthly_ei.values,
    theta=month_names,
    fill='toself',
    name='Emissions Intensity',
    line_color='#C0392B',
    fillcolor='rgba(192,57,43,0.3)',
), row=1, col=2)

fig.update_layout(
    title='Seasonal Rhythm: Clean Energy Share vs Emissions Intensity',
    height=450,
    plot_bgcolor='#FAFAF5', paper_bgcolor='#FAFAF5',
    showlegend=False,
)
fig.show()

print('The two radar charts are mirror images of each other.')
print('When clean energy share peaks (Aug-Sep), emissions intensity is at its lowest.')
print('India\'s grid literally breathes with the monsoon.')

The two radar charts are mirror images of each other.
When clean energy share peaks (Aug-Sep), emissions intensity is at its lowest.
India's grid literally breathes with the monsoon.


## 6. Month-by-Month Emissions Intensity

In [8]:
fig = go.Figure()

fig.add_trace(go.Bar(
    x=month_names, y=monthly_ei.values,
    marker_color=['#C0392B' if v > monthly_ei.mean() else '#27AE60' for v in monthly_ei.values],
    text=[f'{v:.0f}' for v in monthly_ei.values],
    textposition='outside',
    hovertemplate='%{x}: %{y:.0f} tCO2/GWh<extra></extra>',
))

# Mean line
fig.add_hline(y=monthly_ei.mean(), line_dash='dash', line_color='grey',
              annotation_text=f'Annual avg: {monthly_ei.mean():.0f}')

fig.update_layout(
    title='Monthly Emissions Intensity — Red = Above Average, Green = Below',
    yaxis_title='Emissions Intensity (tCO2/GWh)',
    plot_bgcolor='#FAFAF5', paper_bgcolor='#FAFAF5',
    height=400,
)
fig.show()

## Key Findings

1. **India's grid has four distinct seasons:**
   - **Winter (Jan-Feb):** Coal dominates (~76%), lowest clean share (~18%), dirtiest grid
   - **Pre-Monsoon/Summer (Mar-May):** Demand surges with heat, coal maxes out, solar at its best
   - **Monsoon (Jun-Sep):** The great handoff — hydro triples, wind quadruples, coal share drops to ~65-68%, cleanest grid
   - **Post-Monsoon (Oct-Dec):** Transition back to coal dominance, hydro/wind fade

2. **The monsoon handoff is the story:** Coal drops ~10 percentage points in share during monsoon. Hydro goes from ~200 to ~700 MU/day. Wind from ~50 to ~400+.

3. **Solar is surprisingly stable across seasons** — its worst month is still >70% of its peak. It's the one source that doesn't dramatically shift with the monsoon.

4. **The radar charts are mirror images:** Clean energy share and emissions intensity are seasonally anticorrelated. India's grid breathes.

5. **For the art:** The seasonal rhythm is the strongest visual pattern. Any art form (spiral, strip, bloom) should make this monsoon handoff visible.

→ Next: Notebook 05 — Renewable Share